# Train spatial/non-spatial classifier

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
SEED = 42
np.random.seed(SEED)

## ✂ Split our manually labelled dataset into train/val/test (80/10/10)

In [3]:
data = (
    pd.read_excel('interim/is_spatial_20260125_edits.xlsx')
    .dropna(subset=['human_check'])
    .assign(
        y=lambda df_: df_.human_check.astype(int)
    )
    .filter(['query_id', 'query_text', 'y'])
)

print(data.y.value_counts())

data

y
0    755
1    718
Name: count, dtype: int64


,query_id,query_text,y
0,1135449,drugs that may increase homicidal thoughts,0
1,773756,what is motionless,0
2,349892,how to choose a diet plan,0
3,477401,population of carlsbad,0
4,1010534,which herb can heal bladder,0
...,...,...,...
11844,611868,what county is reynolds nd,1
11854,813149,what is the county for kermit tx,1
11898,1167689,weather in north dakota in may,1
11963,1163879,"what county is harvard, ma in/",1


In [4]:
# Split original data into train + test
train_val_df, test_df = train_test_split(
    data,
    test_size=0.1,
    random_state=SEED,
    stratify=data['y'] # ensure equal number of spatial/non-spatial
)

# Now let's split original train into train + val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.11111, # ca 10% of the original set
    random_state=SEED,
    stratify=train_val_df['y']
)

In [5]:
print('==Train counts==')
print(train_df.y.value_counts())

print('\n\n==Val counts==')
print(val_df.y.value_counts())

print('\n\n==Test counts==')
print(test_df.y.value_counts())

==Train counts==
y
0    603
1    574
Name: count, dtype: int64


==Val counts==
y
0    76
1    72
Name: count, dtype: int64


==Test counts==
y
0    76
1    72
Name: count, dtype: int64


In [6]:
test_df.y.value_counts()

y
0    76
1    72
Name: count, dtype: int64

In [7]:
train_df.to_csv('output/classifier.train.csv', index=False)
val_df.to_csv('output/classifier.val.csv', index=False)
test_df.to_csv('output/classifier.test.csv', index=False)

# 💪 Train our classifier

In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from setfit import Trainer, TrainingArguments, SetFitModel
from transformers import EarlyStoppingCallback
import torch

In [13]:
train_ds = Dataset.from_pandas(
    pd.read_csv('output/classifier.train.csv', usecols=['query_text', 'y'])
)

val_ds = Dataset.from_pandas(
    pd.read_csv('output/classifier.val.csv', usecols=['query_text', 'y'])
)

test_ds  = Dataset.from_pandas(
    pd.read_csv('output/classifier.test.csv', usecols=['query_text', 'y'])
)

In [14]:
model = SetFitModel.from_pretrained(
    'BAAI/bge-small-en-v1.5',
    num_pairs=100,
    num_iterations=10
)

# warning of initialising classification head with random weights is OK and expected!

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [16]:
args = TrainingArguments(
    num_epochs=5,
    batch_size=64,
    body_learning_rate=2e-5,
    seed=SEED,
    eval_strategy='steps',
    eval_steps=2000,
    save_strategy='steps',
    save_steps=2000,
    save_total_limit=10,
    load_best_model_at_end=True,
    metric_for_best_model='embedding_loss',
    greater_is_better=False
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    column_mapping={'query_text': 'text', 'y': 'label'},
    metric=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# disable pin_memory in the internal HF trainer
trainer.st_trainer.args.dataloader_pin_memory = False

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/1177 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

***** Running training *****
  Num unique pairs = 694262
  Batch size = 64
  Num epochs = 5


Step,Training Loss,Validation Loss


In [25]:
results = trainer.evaluate(test_ds)
print(f"Accuracy: {results['accuracy']:.3f}")
print(f"F1: {results['f1']:.3f}")

Applying column mapping to the evaluation dataset
***** Running evaluation *****


Accuracy: 0.950
F1: 0.947


In [26]:
trainer.model.save_pretrained("spatial_classifier")

### Sence check: let's manually test on a few queries

In [27]:
model = SetFitModel.from_pretrained("spatial_classifier")

In [28]:
def is_spatial(q):
    pred = model.predict([q])
    print(f'{q}: {"✅" if pred[0] else "🚫"} \n---')

In [29]:
is_spatial("restaurants near me")
is_spatial("don't get near me")
is_spatial("how far ahead should i plan my future")
is_spatial("how long is a piece of string")
is_spatial("how far is york")
is_spatial("what's the longest distance between a city and the sea")
is_spatial("do long-distance relationships work")
is_spatial("how to clean a carpet")
is_spatial("distance between melbourne and london")
is_spatial("where should i keep my money")
is_spatial("where is a bank nearby")

restaurants near me: ✅ 
---
don't get near me: 🚫 
---
how far ahead should i plan my future: 🚫 
---
how long is a piece of string: 🚫 
---
how far is york: ✅ 
---
what's the longest distance between a city and the sea: ✅ 
---
do long-distance relationships work: 🚫 
---
how to clean a carpet: 🚫 
---
distance between melbourne and london: ✅ 
---
where should i keep my money: 🚫 
---
where is a bank nearby: ✅ 
---


# Classify *all* MS MARCO queries

- Runs about 10 min (on an M3 processor)

In [30]:
from setfit import SetFitModel
import pandas as pd
from tqdm import tqdm

In [31]:
model = SetFitModel.from_pretrained("spatial_classifier")
queries = pd.read_csv("interim/queries.csv.zip")

# Define batch prediction function
def batch_predict(texts, batch_size=1024):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        preds.extend(model.predict(batch))
    return preds.item()

# Run prediction
queries['is_spatial_pred'] = batch_predict( queries['query_text'].tolist() )

100%|█████████████████████████████████████████| 988/988 [10:53<00:00,  1.51it/s]


In [42]:
queries

,query_id,query_text,is_spatial_pred
0,1048578,cost of endless pools/swim spa,0
1,1048579,what is pcnt,0
2,1048580,what is pcb waste,0
3,1048581,what is pbis?,0
4,1048582,what is paysky,0
...,...,...,...
1010911,633855,what does canada post regulations mean,0
1010912,1059728,wholesale lularoe price,0
1010913,210839,how can i watch the day after,0
1010914,908165,what to use instead of pgp in windows,0


In [38]:
#queries.is_spatial_pred = queries.is_spatial_pred.apply(lambda tsr: tsr.item())

In [41]:
queries.to_csv('output/queries-classified.csv.zip', index=False)